---

MRI analysis 

---

In [1]:
# autoload
%load_ext autoreload
%autoreload 2
from pgl import pgl, pglDatabase, pglActions, pglFlywheel
import os
from pathlib import Path
from uuid import uuid4

---

Load behavioral data

---

In [ ]:
pgl.importSettings()

In [ ]:
newSession = pglActions.loadSession.execute()
if newSession: session = newSession

---

Load data from flywheel

---

In [ ]:
# login to flywheel with apikey
apikey = os.environ['FW_API_KEY']
flywheel = pglFlywheel(apikey=apikey)

In [ ]:
cachePath = Path("~/data/deleteme").expanduser() / f"flywheel-action-test-{uuid4().hex}"
loadFlywheelSession = pglActions.loadFlywheelSession()
loadFlywheelSession.configure(flywheel=flywheel, cacheDirectory=cachePath, experimentName="Flywheel action integration test", choose=True)

In [ ]:
session.print()

In [ ]:
result.provenance["invocation"]

In [2]:
def testAFNIUserScriptPreparation(sourceFilename):
    """Test necessary relocation checks without installing anything."""
    import hashlib
    import tempfile
    from pathlib import Path
    from unittest.mock import patch

    from pgl.pglExternalTools import pglAFNIInstallation

    sourceText = Path(sourceFilename).expanduser().read_text(encoding="utf-8")

    with tempfile.TemporaryDirectory(prefix="pgl-user-script-test-") as directory:
        root = Path(directory) / "afni"
        installation = pglAFNIInstallation(root)

        with patch("subprocess.run") as launch:
            prepared = installation.prepareUserScript(sourceText)

        launch.assert_not_called()
        assert not root.exists()

        executableDirectory = str(installation.root / "abin")
        buildDirectory = str(installation.root / "build")
        script = prepared["scriptText"]

        assert f"-bindir {executableDirectory}" in script
        assert f"-dir_bin {executableDirectory}" in script
        assert f"-build_root {buildDirectory}" in script
        assert f"-abin {executableDirectory}" in script
        assert f"set path = ( {executableDirectory} $path )" in script
        assert f"{executableDirectory}/build_afni.py " in script

        # Preserve ordinary setup operations and user configuration.
        for text in (
            "defaults write com.apple.Terminal FocusFollowsMouse -string YES",
            "setenv R_LIBS $HOME/sw/R-$rver",
            "rPkgsInstall -pkgs ALL",
            "suma -update_env",
            "apsearch -update_all_afni_help",
            "afni_system_check.py -check_all",
        ):
            assert text in script

        assert prepared["sourceIdentity"]["sha256"] == hashlib.sha256(sourceText.encode("utf-8")).hexdigest()
        assert prepared["preparedIdentity"]["sha256"] == hashlib.sha256(script.encode("utf-8")).hexdigest()
        assert installation.prepareUserScript(sourceText) == prepared

        # Comments, spacing, and unrelated compiler options may change.
        changed = sourceText + "\n# Another note about ~/abin and ~/afni_build.\n"
        changed = changed.replace("-cc_path /usr/bin/gcc", "-cc_path /another/compiler")
        changed = changed.replace("-build_root ", "-build_root    ")

        accepted = installation.prepareUserScript(changed)
        assert "-cc_path /another/compiler" in accepted["scriptText"]

        # A destination that the adapter cannot relocate must stop preparation.
        changed = sourceText.replace("-bindir $HOME/abin", "-bindir /another/installation")

        try:
            installation.prepareUserScript(changed)
        except ValueError:
            pass
        else:
            raise AssertionError("An unrelated bootstrap destination was accepted.")

        # Two build invocations are ambiguous for this installation procedure.
        changed = sourceText + "\nbuild_afni.py -build_root ~/afni_build\n"

        try:
            installation.prepareUserScript(changed)
        except ValueError:
            pass
        else:
            raise AssertionError("Ambiguous build commands were accepted.")

        assert not root.exists()

    print("AFNI user-script preparation tests passed.")

In [3]:
testAFNIUserScriptPreparation("~/proj/pgl/OS_notes.macos_12_b_user.tcsh")

AFNI user-script preparation tests passed.


In [ ]:
def testAFNIUserStageWorker():
    """Test user-stage execution/state using simulated subprocesses only."""
    import hashlib
    import json
    import tempfile
    from pathlib import Path
    from types import SimpleNamespace
    from unittest.mock import patch
    from uuid import uuid4

    from pgl.pglExternalTools import pglAFNIInstallation

    def prepare(root):
        stageDirectory = root / "install" / "user-test"
        stageDirectory.mkdir(parents=True)
        executableDirectory = root / "abin"
        executableDirectory.mkdir()

        # Files only establish availability; subprocess execution is mocked.
        for name in ("afni", "3dvolreg", "afni_system_check.py"):
            path = executableDirectory / name
            path.write_text("# Test fixture; never executed.\n")
            path.chmod(0o700)

        script = b"# Test fixture; never executed.\necho test\n"
        (stageDirectory / "user-setup.tcsh").write_bytes(script)

        statePath = root / "install" / "state.json"
        installationID = str(uuid4())
        state = {
            "schemaVersion": 1,
            "tool": "AFNI",
            "installationID": installationID,
            "installPath": str(root.resolve()),
            "status": "userLaunchPending",
        }
        statePath.write_text(json.dumps(state))

        request = {
            "schemaVersion": 1,
            "approved": True,
            "prerequisitesConfirmed": True,
            "installationID": installationID,
            "statePath": str(statePath),
            "stageDirectory": str(stageDirectory),
            "scriptIdentity": {
                "sha256": hashlib.sha256(script).hexdigest(),
                "size_bytes": len(script),
            },
            "environmentRemove": ["PYTHONPATH"],
            "environmentOverrides": {"PATH": "/usr/bin:/bin:/usr/sbin:/sbin"},
        }
        requestPath = stageDirectory / "request.json"
        requestPath.write_text(json.dumps(request))
        return requestPath, statePath, stageDirectory

    def successfulCommand(arguments, **kwargs):
        output = kwargs.get("stdout")
        if hasattr(output, "write"): output.write(b"Simulated diagnostic output.\n")
        return SimpleNamespace(returncode=0)

    with tempfile.TemporaryDirectory(prefix="pgl-user-worker-test-") as directory:
        directory = Path(directory)

        with (
            patch("platform.system", return_value="Darwin"),
            patch("platform.machine", return_value="arm64"),
        ):
            # Successful scripts/probes require review, not automatic approval.
            requestPath, statePath, stageDirectory = prepare(directory / "success")

            with patch("subprocess.run", side_effect=successfulCommand) as run:
                assert pglAFNIInstallation.runUserStage(requestPath) == 0

            state = json.loads(statePath.read_text())
            assert state["status"] == "verificationPending"
            assert len(state["userSteps"]) == 4
            assert run.call_count == 4

            # Verification must use this installation, not a command from PATH.
            for call in run.call_args_list[1:]:
                executable = Path(call.args[0][0])
                assert executable.parent.resolve() == (directory / "success" / "abin").resolve()

            # Script failure stops before any verification commands.
            requestPath, statePath, _ = prepare(directory / "failure")

            with patch("subprocess.run", return_value=SimpleNamespace(returncode=7)) as run:
                assert pglAFNIInstallation.runUserStage(requestPath) == 1

            assert run.call_count == 1
            assert json.loads(statePath.read_text())["status"] == "userFailed"

            # Changed approved bytes must prevent execution entirely.
            requestPath, statePath, stageDirectory = prepare(directory / "changed")
            (stageDirectory / "user-setup.tcsh").write_text("changed after approval\n")

            with patch("subprocess.run") as run:
                assert pglAFNIInstallation.runUserStage(requestPath) == 1

            run.assert_not_called()
            assert json.loads(statePath.read_text())["status"] == "userFailed"

            # An interruption is preserved and propagated.
            requestPath, statePath, _ = prepare(directory / "interrupt")

            with patch("subprocess.run", side_effect=KeyboardInterrupt):
                try:
                    pglAFNIInstallation.runUserStage(requestPath)
                except KeyboardInterrupt:
                    pass
                else:
                    raise AssertionError("KeyboardInterrupt was not propagated.")

            assert json.loads(statePath.read_text())["status"] == "userInterrupted"

    print("AFNI user-stage worker tests passed.")

In [2]:
testAFNIUserStageWorker()


=== Running approved AFNI user setup ===
Working directory: /private/var/folders/n2/4bqdpl1s6wd9x4l9ll4p03_m0000gq/T/pgl-user-worker-test-pjjhax8a/success/install/user-test
AFNI executable directory: /private/var/folders/n2/4bqdpl1s6wd9x4l9ll4p03_m0000gq/T/pgl-user-worker-test-pjjhax8a/success/abin

=== Checking afni ===

=== Checking 3dvolreg ===

=== Checking afni_system_check.py ===

AFNI user setup and executable probes completed.
Installation is awaiting diagnostic review—not yet marked verified.
System check: /private/var/folders/n2/4bqdpl1s6wd9x4l9ll4p03_m0000gq/T/pgl-user-worker-test-pjjhax8a/success/install/user-test/systemCheck.log
Review its recommendations and test AFNI/SUMA GUI operation.

Then reopen Python and run:
    afni.install(installPath='/private/var/folders/n2/4bqdpl1s6wd9x4l9ll4p03_m0000gq/T/pgl-user-worker-test-pjjhax8a/success')


AssertionError: 